# ② 応用ハンズオン【演習版】
**岡山県 高校教員向け データサイエンス・ハンズオン（2時間目）**

1時間目で学んだことを使って、**自分で手を動かして**分析します。
`____` の部分を自分で埋めて実行しましょう（解答版もあります）。

テーマ：**生活習慣と成績の関係を調べる**
- `class_sample.csv` … ある高校の生徒360人ぶんの**練習用サンプル（架空データ）**
- `gakuryoku_lifestyle.csv` … 全国学力・学習状況調査にもとづく**生活習慣と正答率**の整理データ


## 0. 準備

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os

# データの入手方法：
#  ・手動アップロード（既定）：BASE_URL は "" のまま。実行すると選択画面が出るのでCSVを選ぶ。
#  ・公開リポジトリを使う場合のみ：BASE_URL に raw URL を入れる
#    例) "https://raw.githubusercontent.com/USER/REPO/main/data/"
#    ※ Privateリポジトリの raw URL はトークンが必要なため受講者配布には不向き。手動アップロードを推奨。
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    """data/名前.csv を読み込む。ローカル→(URL)→手動アップロードの順に試す。"""
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload()
    key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))

print("準備OK：load('ファイル名.csv') でデータを読み込めます")


In [ ]:
# 日本語をグラフに表示できるようにする（Colabでは最初に1回だけ）
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib   # これでグラフの日本語が文字化けしない
print("日本語フォント準備OK")


## 1. データを読み込んで、ざっと眺める

In [ ]:
df = load("class_sample.csv")
df.head()


In [ ]:
# 行数・列数と、各列の要約を見てみよう
print(df.shape)
df.____()   # ヒント：要約統計量を出すメソッド


In [ ]:
# 欠損値（空っぽのデータ）がないか確認
df.isnull().sum()


## 2. 群で比べる：朝食を食べる人 / 食べない人の平均点

In [ ]:
# breakfast（1=食べる, 0=食べない）ごとに test_score の平均を出す
grp = df.groupby("____")["test_score"].____()
print(grp)

In [ ]:
# 箱ひげ図で分布を比べる
df.boxplot(column="____", by="____", figsize=(6,4))
plt.title("朝食の有無と点数"); plt.suptitle(""); plt.show()

## 3. 相関：勉強時間と点数の関係

In [ ]:
# 散布図：横軸=study_min, 縦軸=test_score
plt.figure(figsize=(6,5))
plt.scatter(df["____"], df["____"], alpha=0.4)
plt.xlabel("1日の勉強時間（分）"); plt.ylabel("テストの点数")
plt.title("勉強時間と点数"); plt.show()

### 相関係数と「相関の強さ」の目安
2つの量が「いっしょに増減する関係」の強さを、1つの数字で表したものが **相関係数 r** です。

- 範囲は **-1 〜 +1**。
- **符号**：`+`＝片方が増えるともう片方も増える（正の相関）／`-`＝片方が増えるともう片方は減る（負の相関）。
- **絶対値 |r| の大きさ＝関係の強さ**。0に近いほど「関係がうすい」。

| \|r\| の値 | 相関の強さ（目安） |
|---|---|
| 0.0 〜 0.2 | ほとんど相関なし |
| 0.2 〜 0.4 | 弱い相関 |
| 0.4 〜 0.7 | 中くらいの相関 |
| 0.7 〜 1.0 | 強い相関 |

**注意点（大事）**
- 相関係数は **直線的な関係** の強さ。曲線の関係はうまく測れない。
- **外れ値**（極端な点）に引っ張られやすい → 必ず散布図もいっしょに見る。
- **相関≠因果**：関係があっても「原因」とは限らない（このあとの章で扱います）。


In [ ]:
# 相関係数（-1〜1。1に近いほど強い正の相関）
print("相関係数：", df["study_min"].____(df["test_score"]))

In [ ]:
# 数値の列すべての相関を一気に見る（相関行列 → ヒートマップ）
import numpy as np
num = df[["height_cm","weight_kg","study_min","sleep_hours","smartphone_hours","test_score"]]
corr = num.corr()
plt.figure(figsize=(6,5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("相関行列")
plt.tight_layout(); plt.show()
corr.round(2)


## 4. 単回帰：勉強時間から点数を予測する

In [ ]:
import numpy as np
x = df["study_min"]; y = df["test_score"]
a, b = np.polyfit(x, y, 1)     # y = a*x + b  （直線をあてはめる）
print(f"傾き={a:.3f}, 切片={b:.1f}")
print(f"→ 勉強を10分増やすと、点数は約 {a*10:.2f} 点上がる（このデータ上では）")

# 散布図に回帰直線を重ねる
plt.figure(figsize=(6,5))
plt.scatter(x, y, alpha=0.3)
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, a*xx + b, color="red", linewidth=2)
plt.xlabel("勉強時間（分）"); plt.ylabel("点数"); plt.title("単回帰")
plt.show()

## 5. 全国データで見る：生活習慣と正答率

In [ ]:
g = load("gakuryoku_lifestyle.csv")
g.head(20)


In [ ]:
# category が「朝食」の行だけ取り出して棒グラフに
asa = g[g["category"] == "____"]
plt.figure(figsize=(7,4))
plt.bar(asa["group"], asa["math_score"])
plt.title("朝食習慣と数学の平均正答率（全国）")
plt.ylabel("平均正答率(%)"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 6. 考えよう：相関 ≠ 因果
- 「朝食を食べる子ほど正答率が高い」…では**朝食を食べさせれば成績が上がる**と言える？
- 背景に別の要因（生活リズム全体、家庭環境など）が隠れているかも（**交絡**）。
- データ分析は「関係がありそう」を見つける道具。**因果の証明には別の工夫が必要**。

> この議論こそ、生徒に伝えたいデータリテラシーの核心です。

### まとめ（2時間目）
群間比較・相関・回帰・相関行列を、自分の手で実行できた。
次（3時間目）は、**グループでテーマを決めて自由に分析**します。